# worldgen quickstart

This notebook generates a planet from a specification, prints its report, draws maps in several projections, and saves and reloads the world.

Run it from the repository root after `pip install -e ".[dev,notebooks]"`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from worldgen import PlanetSpec, generate_world, format_report, load_world, save_world
from worldgen.priors import ARCHETYPES
from worldgen.render import animate_history, plot_climate, plot_history, plot_map, plot_overview, projection
import tempfile
from pathlib import Path

## 1. Available archetypes

Archetypes are presets for recognisable kinds of world. Anything you leave unset in a spec is drawn from the chosen archetype.

In [ ]:
for a in ARCHETYPES.values():
    print(f"{a.name:<16}{a.description}")

## 2. Generate a planet

Numbers fix a value, `(low, high)` tuples draw from a range, and omitted values are drawn or derived. `resolution` is `"preview"` (10k cells), `"standard"` (40k) or `"high"` (160k).

`snapshot_interval_myr` keeps the plate-tectonic history (section 4).

`body.water_mass_fraction` is the planet's total water. Part of it is held in the mantle; the rest fills the ocean basins and sets how much land is left. Setting `surface.land_fraction` instead fixes the land and adjusts the water.

In [ ]:
spec = PlanetSpec(
    name="Kestrel",
    seed=11,
    priors={"archetype": "temperate"},
    star={"mass_msun": (0.85, 1.0)},
    body={"mass_mearth": 1.3, "water_mass_fraction": 5e-4},
)
world = generate_world(spec, resolution="standard", snapshot_interval_myr=40)
print(format_report(world.state))

## 3. Maps

`plot_overview` shows elevation, landform classes, plates and two globe views. `plot_map` draws one field in any projection.

In [ ]:
fig = plot_overview(world)

In [ ]:
from worldgen.render import projection

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 2, 1, projection=projection("robinson"))
plot_map(world, "orogeny_age", "robinson", ax=ax1)
ax2 = fig.add_subplot(1, 2, 2, projection=projection("north_polar"))
plot_map(world, "elevation", "north_polar", ax=ax2, title="North polar view")

## 4. Tectonic history

Mobile-lid planets get their continents from a plate simulation (400 Myr by default). `plot_history` shows the snapshots; `animate_history` writes them as a GIF. The report's `features` line summarises the run: plates, rifts, merges, subducted crust and the median sea-floor age.

In [ ]:
fig = plot_history(world, panels=6)

In [ ]:
from IPython.display import Image

gif = animate_history(world, Path(tempfile.mkdtemp()) / "kestrel.gif", fps=3, width=600)
Image(filename=gif)

The start (`supercontinent` or `cratons`) and the duration can be set in the spec. `tectonics: heuristic` skips the simulation for a fast layout.

In [ ]:
fig = plt.figure(figsize=(15, 4))
variants = [("supercontinent, 150 Myr", {"tectonics_start": "supercontinent", "tectonics_duration_myr": 150}),
            ("cratons, 600 Myr", {"tectonics_start": "cratons", "tectonics_duration_myr": 600}),
            ("heuristic", {"tectonics": "heuristic"})]
for k, (label, surface) in enumerate(variants):
    w = generate_world(spec.model_copy(update={"surface": spec.surface.model_copy(update=surface)}), "preview")
    ax = fig.add_subplot(1, 3, k + 1, projection=projection("mollweide"))
    plot_map(w, "elevation", ax=ax, colorbar=False, title=label)

## 5. Water, rivers and lakes

Planets with liquid surface water get rivers, lakes and river erosion, driven by the climate model (section 6). `rainfall` and `basins` are map fields; rivers are drawn over elevation, terrain, rainfall and basin maps. Brown-tinted basins drain to lakes with no outlet to the sea.

In [ ]:
w = world.state.water
print(f"surface water: {w.surface_mass_kg / 1.4e21:.2f} Earth oceans, mantle: {w.mantle_mass_fraction * world.state.bulk.mass_kg / 1.4e21:.2f}")
fig = plt.figure(figsize=(15, 5))
for k, field in enumerate(["rainfall", "basins"]):
    ax = fig.add_subplot(1, 2, k + 1, projection=projection("robinson"))
    plot_map(world, field, "robinson", ax=ax, legend="inside")

In [ ]:
ds = world.surface
land = ~ds["ocean"].values
mouths = land & ds["ocean"].values[ds["flow_to"].values]
print(f"largest river mouths (m³/s): {np.sort(ds['discharge'].values[mouths])[::-1][:5].round(-2)}")
print(f"land under lakes: {ds['lake'].values[land].mean():.1%}, draining inland: {ds['endorheic'].values[land].mean():.0%}")
ax = plot_map(world, "elevation", "orthographic", central_longitude=40, central_latitude=15)

## 6. Climate, ice and life

The global state uses a zonal energy-balance model with seasons (Tier 0) for the mean temperature, planetary albedo, ice line and open water. The surface gets a two-dimensional energy-balance model with moisture transport (Tier 1), iterated with ice sheets and vegetation. `plot_climate` shows annual and seasonal temperature and precipitation, biomes, Köppen–Geiger classes and zonal means. Month 1 begins at the northern winter solstice.

In [ ]:
s = world.state
print(s.climate)
print(s.biosphere)
fig = plot_climate(world)

In [ ]:
fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 2, 1, projection=projection("north_polar"))
plot_map(world, "ice", "north_polar", ax=ax1, title="Ice sheets and sea ice")
ax2 = fig.add_subplot(1, 2, 2, projection=projection("robinson"))
plot_map(world, "temperature", "robinson", ax=ax2, month=7, title="Temperature, month 7")
ds = world.surface
land = ~ds["ocean"].values
print(f"vegetated land: {ds['vegetation_cover'].values[land].mean():.0%}, "
      f"ice sheets: {(ds['ice_thickness'].values[land] > 0).mean():.1%} of land")

Vegetation colour follows the star's light: pigments absorb near the peak of the surface photon flux (Kiang et al. 2007), so the colour and albedo of plants depend on the star.

In [ ]:
from worldgen.biosphere.pigment import absorption_peak, pigment_colour
for teff in (7000, 5772, 4500, 3200):
    colour, brightness = pigment_colour(absorption_peak(teff), teff)
    print(f"{teff} K star: pigment peak {absorption_peak(teff):.0f} nm, vegetation {colour}, albedo {brightness:.2f}")

## 7. Working with the surface data

The surface is an `xarray.Dataset` on the grid cells, with latitude and longitude coordinates.

In [ ]:
ds = world.surface
print(ds)
elevation = ds["elevation"].values
ocean = ds["ocean"].values
print(f"land fraction: {1 - ocean.mean():.2f}")
print(f"highest point: {elevation.max():.0f} m, deepest point: {elevation.min():.0f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(elevation / 1e3, bins=80, color="0.3")
ax.axvline(0, color="tab:blue", lw=1)
ax.set_xlabel("elevation (km)")
ax.set_ylabel("cells")
ax.set_title("Hypsometry: continents and ocean floor form two peaks")

## 8. Comparing archetypes

In [ ]:
names = ["ocean", "arid", "ice", "tidally_locked", "hot_volcanic", "small_world"]
fig = plt.figure(figsize=(15, 6.5))
for k, name in enumerate(names):
    w = generate_world(PlanetSpec(name=name, seed=10 + k, priors={"archetype": name}), resolution="preview")
    ax = fig.add_subplot(2, 3, k + 1, projection=projection("mollweide"))
    s = w.state
    plot_map(w, "elevation", "mollweide", ax=ax, colorbar=False, width=500,
             title=f"{name}: {s.interior.tectonic_regime.replace('_', ' ')}, "
                   f"{s.atmosphere.surface_temperature_k:.0f} K")

## 9. History mode: where the planet came from

`mode="history"` integrates the planet from formation instead of evaluating relations at one epoch: the star's brightening and XUV, the mantle and core cooling with their dynamo, outgassing and escape, the water exchanged with the mantle, the carbonate-silicate cycle under a Tier 0 climate, and the biosphere. `generate` then returns a timeline as well as the state: the events that happened, full states at the epochs the spec asks for, and the sampled series behind the figures.


In [ ]:
from worldgen import format_timeline, generate
from worldgen.render import plot_climate_history, plot_interior_history, plot_life_history

earth = PlanetSpec(
    name="Earth (history)",
    seed=0,
    mode="history",
    star={"mass_msun": 1.0, "age_gyr": 4.57, "activity_percentile": 0.5},
    orbit={"semi_major_axis_au": 1.0, "eccentricity": 0.0167, "obliquity_deg": 23.44,
           "rotation_period_h": 23.934},
    body={"mass_mearth": 1.0, "core_mass_fraction": 0.325},
    interior={"tectonic_regime": "mobile_lid"},
    history={"epochs_gyr": [1.0, 2.5, 4.0], "initial_water_mass_fraction": 0.00064},
)
state, timeline = generate(earth)
print(format_timeline(timeline, state))

Each figure draws the same run from a different side. The events are marked on every panel and the stretches the planet spent frozen, in a runaway or dry are shaded.

In [ ]:
fig = plot_climate_history(timeline)

In [ ]:
fig = plot_life_history(timeline)

The history also reaches the surface. The plate simulation takes its speed from the plate creation rate the thermal model produces and its volcanism from the melt, and the span it simulates follows the sea-floor turnover. What the planet used to be leaves relicts, for as long as its own erosion preserves them: Earth reworks its surface in ~100 Myr and keeps none, while a cold, quiet world keeps the shoreline of a sea it has lost, valley networks cut when it still had rain, and ground the ice has left.

In [ ]:
relict_world = generate_world(
    PlanetSpec(name="Relict world", seed=5, mode="history", priors={"archetype": "ice"}),
    resolution="preview")
print({k: round(v, 3) for k, v in relict_world.state.surface.features.items() if k.startswith("relict")})
fig = plot_map(relict_world, "relicts", "mollweide", width=700).figure

## 10. Save and reload

A saved world is a folder with `spec.yaml`, `state.yaml` and `surface.zarr`, plus `timeline.yaml` and an `epochs/` folder in history mode. The same folder works with the command line (`worldgen map`, `worldgen report`, `worldgen drift`, `worldgen history`).

In [ ]:
folder = save_world(world, Path(tempfile.mkdtemp()) / "kestrel")
reloaded = load_world(folder)
print(sorted(p.name for p in folder.iterdir()))
print(np.array_equal(reloaded.surface["elevation"].values, world.surface["elevation"].values))